# Load & Compress Data Pipeline

This notebook replicates the exact data loading and transformation pipeline used by the geospatial site scoring application.

**Pipeline stages:**
1. **Load** all 7 input CSV files (same as `data_transform.py`)
2. **Aggregate** monthly records → one row per site
3. **Relative strength** momentum indicators (multi-horizon)
4. **Join** geospatial distance features

**Input:** 7 CSV files totaling ~1 GB  
**Output:** `site_aggregated_precleaned.parquet` (~12 MB, 57,675 sites × 102 columns)

In [10]:
# Inspect input directory files
import polars as pl
import numpy as np
from pathlib import Path
from datetime import datetime, date
from typing import Optional, List, Tuple

# Project paths
PROJECT_ROOT = Path('.').resolve()
INPUT_DIR = PROJECT_ROOT / 'data' / 'input'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input directory:  {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'\nInput files:')
for f in sorted(INPUT_DIR.iterdir()):
    if f.is_file() and f.suffix == '.csv':
        size_mb = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:50s} {size_mb:8.1f} MB')

Input directory:  /Users/home/gkdev/geospatial/data/input
Output directory: /Users/home/gkdev/geospatial/data/processed

Input files:
  Sites - Base Data Set.csv                              16.9 MB
  kroger_dist_to_site_mi.csv                              2.1 MB
  mcdonalds_dist_to_site_mi.csv                           7.5 MB
  nearest_site_distances.csv                              6.5 MB
  salesforce_site_revenue.csv                             5.1 MB
  site_interstate_distances.csv                           3.8 MB
  site_kroger_distances.csv                               2.7 MB
  site_mcdonalds_distances.csv                            2.8 MB
  site_proximity_features.csv                             9.4 MB
  site_scores_revenue_and_diagnostics.csv               926.7 MB
  site_target_distances.csv                               2.7 MB
  site_walmart_distances.csv                              2.7 MB
  sites_base_data_set.csv                                16.9 MB


---
## Stage 1: Load All Input Files

The application uses **Polars** (not pandas) for loading because the main CSV is 927 MB / 1.4M rows.
Polars is 10-20x faster than pandas for this workload (ADR-003).

### 1a. Main Site Scores Data

In [11]:
# Load site_scores_revenue_and_diagnostics
print('Loading Site Scores data...')
site_scores = pl.read_csv(
    INPUT_DIR / 'site_scores_revenue_and_diagnostics.csv',
    infer_schema_length=10000,
    null_values=['', 'NA', 'null', 'Unknown']
)
print(f'Loaded {len(site_scores):,} monthly records for {site_scores["id_gbase"].n_unique():,} unique sites')
print(f'Columns ({len(site_scores.columns)}): {site_scores.columns[:10]} ... (showing first 10)')
print(f'Date range: {site_scores["date"].min()} to {site_scores["date"].max()}')
site_scores.head(3)

Loading Site Scores data...
Loaded 1,475,761 monthly records for 57,675 unique sites
Columns (94): ['date', 'id_gbase', 'month', 'year', 'monthly_impressions', 'monthly_nvis', 'avg_daily_impressions', 'avg_daily_nvis', 'avg_latency', 'screen_count'] ... (showing first 10)
Date range: 2022-01-01 to 2025-11-01


date,id_gbase,month,year,monthly_impressions,monthly_nvis,avg_daily_impressions,avg_daily_nvis,avg_latency,screen_count,revenue,site_activated_date,network,state,gtvid,county,latitude,longitude,zip,zip_4,dma,dma_rank,statuis,program,experience_type,hardware_type,retailer,avg_household_income,median_age,index_african_american,pct_african_american,index_asian,pct_asian,index_female,pct_female,index_male,pct_male,…,r_cpg_beverage_wine_video,r_finance_credit_cards,r_cpg_cbd_hemp_ingestibles_non_thc,r_cpg_non_food_beverage_cannabis_medical,r_cpg_non_food_beverage_cannabis_recreational,r_cpg_non_food_beverage_cbd_hemp_non_thc,r_alcohol_drink_responsibly_message,r_alternative_transportation,r_associations_and_npo_anti_smoking,r_automotive_after_market_oil,r_cpg_beverage_spirits_ooh,r_cpg_beverage_spirits_video,r_cpg_non_food_beverage_e_cigarette,r_entertainment_casinos_and_gambling,r_government_political,r_automotive_electric,r_recruitment,r_restaurants_cdr,r_restaurants_qsr,r_retail_automotive_service,r_retail_grocery,r_retail_grocerty_with_fuel,c_emv_enabled,c_nfc_enabled,c_open_24_hours,c_sells_beer,c_sells_diesel_fuel,c_sells_lottery,c_vistar_programmatic_enabled,c_walk_up_enabled,c_sells_wine,brand_fuel,brand_restaurant,brand_c_store,monthly_impressions_per_screen,monthly_nvis_per_screen,monthly_revenue_per_screen
str,str,i64,i64,i64,i64,i64,i64,str,i64,f64,str,str,str,str,str,f64,f64,i64,str,str,i64,str,str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str,str,str,bool,str,str,str,str,str,f64,f64,f64
"""2023-06-01""","""59ec02473055c91000eb2f10""",6,2023,null,null,null,null,null,8,0.0,"""2016-11-12""","""Gilbarco""","""FL""","""TAG017""","""Hamilton County""",30.519837,-83.056566,32052,"""32052-4309""","""Tallahassee-Thomasville FL-GA""",105,"""Active""","""Standard ICS""","""Standard GSTV""","""Flex Pay IV - 10.4 Inch""","""Johnson & Johnson, Inc.""",47534,36.2,315.02,39.73,5.74,0.32,69.845494,35.519896,131.203678,64.480104,…,false,false,false,true,true,false,false,false,false,false,false,false,false,false,false,false,false,true,true,false,false,false,null,"""Yes""","""No""","""Yes""",null,"""Yes""",true,null,"""No""","""Marathon""","""Burger King""","""Busy Bee""",null,null,0.0
"""2023-06-01""","""59ec025b0ce7611100819bb8""",6,2023,2324,2324,77,77,null,8,160.24,"""2016-11-15""","""Gilbarco""","""MD""","""BLG086""","""Baltimore City""",39.346382,-76.609371,21212,"""21212-4730""","""Baltimore MD""",29,"""Active""","""Applause - Verifone Owned""","""Standard GSTV""","""Flex Pay IV - 10.4 Inch""","""Sunoco Inc.""",131819,38.6,310.62,39.17,82.92,4.63,105.868717,53.83949,93.927092,46.16051,…,false,true,true,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,"""Yes""","""No""","""No""",null,"""Yes""",true,null,"""No""","""Sunoco""",null,"""Sunoco""",290.5,290.5,20.03
"""2023-06-01""","""59ec02b80ce7611100819e20""",6,2023,1115,1114,101,101,null,8,9.42,"""2021-08-25""","""Dover""","""VA""","""RLE070""","""Montgomery County""",37.128045,-80.417762,24073,"""24073-3355""","""Roanoke-Lynchburg VA""",70,"""Temporarily Deactivated""","""Dover - IOTV2""","""Standard GSTV""","""Wayne Integrated 10""","""Bandys Inc""",79435,36.8,39.61,4.99,41.69,2.33,101.265791,51.498674,98.690168,48.501326,…,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,"""Yes""","""No""","""Yes""",null,"""Yes""",true,null,"""Yes""","""Sunoco""",null,"""Lucky Stop""",139.375,139.25,1.1775


### 1b. Auxiliary Geospatial Distance Files

Six auxiliary CSVs provide distance-to-landmark features (in miles).

In [12]:
# Load auxiliary geospatial data 
print('Loading auxiliary geospatial data...\n')

# 1. Nearest site distances
nearest = pl.read_csv(INPUT_DIR / 'nearest_site_distances.csv')
print(f'Nearest sites:         {len(nearest):,} records | cols: {nearest.columns}')

# 2. Interstate distances — some sites have multiple, take minimum distance
interstate_raw = pl.read_csv(INPUT_DIR / 'site_interstate_distances.csv')
interstate = interstate_raw.group_by('GTVID').agg([
    pl.col('distance_to_interstate_mi').min().alias('min_distance_to_interstate_mi'),
    pl.col('nearest_interstate').first().alias('nearest_interstate')
])
print(f'Interstate distances:   {len(interstate):,} unique sites (from {len(interstate_raw):,} raw records)')

# 3. Kroger distances (pre-aggregated — one row per site)
kroger = pl.read_csv(INPUT_DIR / 'site_kroger_distances.csv')
print(f'Kroger distances:      {len(kroger):,} records')

# 4. McDonald's distances (pre-aggregated — one row per site)
mcdonalds = pl.read_csv(INPUT_DIR / 'site_mcdonalds_distances.csv')
print(f"McDonald's distances:  {len(mcdonalds):,} records")

# 5. Walmart distances
walmart = pl.read_csv(INPUT_DIR / 'site_walmart_distances.csv')
print(f'Walmart distances:     {len(walmart):,} records')

# 6. Target distances
target = pl.read_csv(INPUT_DIR / 'site_target_distances.csv')
print(f'Target distances:      {len(target):,} records')

Loading auxiliary geospatial data...

Nearest sites:         67,644 records | cols: ['GTVID', 'Latitude', 'Longitude', 'nearest_site', 'nearest_site_lat', 'nearest_site_lon', 'nearest_site_distance_m', 'nearest_site_distance_mi']
Interstate distances:   67,644 unique sites (from 70,300 raw records)
Kroger distances:      57,675 records
McDonald's distances:  57,675 records
Walmart distances:     57,675 records
Target distances:      57,675 records


---
## Stage 2: Aggregate Monthly Records → One Row Per Site

The raw data has one row per site per month. We aggregate to one row per site with:
- **Total** revenue and auction metrics across all months
- **Average** metrics per active month
- **Most recent** metadata values (status, location, demographics, etc.)

In [13]:
# Define revenue_metrics and agg_exprs
revenue_metrics = [
    'revenue', 'monthly_impressions', 'monthly_nvis',
    'monthly_impressions_per_screen', 'monthly_nvis_per_screen',
    'monthly_revenue_per_screen'
]

# Build aggregation expressions — mirrors data_transform.aggregate_site_metrics()
agg_exprs = [
    # Count of active months
    pl.len().alias('active_months'),
    # Date range
    pl.col('date').min().alias('first_month'),
    pl.col('date').max().alias('last_month'),
    # Most recent site metadata (last value when sorted by date)
    pl.col('gtvid').last().alias('gtvid'),
    pl.col('site_activated_date').last().alias('site_activated_date'),
    pl.col('network').last().alias('network'),
    pl.col('state').last().alias('state'),
    pl.col('county').last().alias('county'),
    pl.col('latitude').last().alias('latitude'),
    pl.col('longitude').last().alias('longitude'),
    pl.col('zip').last().alias('zip'),
    pl.col('dma').last().alias('dma'),
    pl.col('dma_rank').last().alias('dma_rank'),
    pl.col('statuis').last().alias('status'),  # Note: typo in source data ('statuis' not 'status')
    pl.col('program').last().alias('program'),
    pl.col('experience_type').last().alias('experience_type'),
    pl.col('hardware_type').last().alias('hardware_type'),
    pl.col('retailer').last().alias('retailer'),
    pl.col('screen_count').last().alias('screen_count'),
    # Demographics
    pl.col('avg_household_income').last().alias('avg_household_income'),
    pl.col('median_age').last().alias('median_age'),
    pl.col('pct_african_american').last().alias('pct_african_american'),
    pl.col('pct_asian').last().alias('pct_asian'),
    pl.col('pct_female').last().alias('pct_female'),
    pl.col('pct_male').last().alias('pct_male'),
    pl.col('pct_hispanic').last().alias('pct_hispanic'),
    # Brands
    pl.col('brand_fuel').last().alias('brand_fuel'),
    pl.col('brand_restaurant').last().alias('brand_restaurant'),
    pl.col('brand_c_store').last().alias('brand_c_store'),
    # Capability flags
    pl.col('schedule_site').last().alias('schedule_site'),
    pl.col('sellable_site').last().alias('sellable_site'),
    pl.col('c_emv_enabled').last().alias('c_emv_enabled'),
    pl.col('c_nfc_enabled').last().alias('c_nfc_enabled'),
    pl.col('c_open_24_hours').last().alias('c_open_24_hours'),
    pl.col('c_sells_beer').last().alias('c_sells_beer'),
    pl.col('c_sells_diesel_fuel').last().alias('c_sells_diesel_fuel'),
    pl.col('c_sells_lottery').last().alias('c_sells_lottery'),
    pl.col('c_vistar_programmatic_enabled').last().alias('c_vistar_programmatic_enabled'),
    pl.col('c_walk_up_enabled').last().alias('c_walk_up_enabled'),
    pl.col('c_sells_wine').last().alias('c_sells_wine'),
    # Restriction flags
    pl.col('r_lottery').last().alias('r_lottery'),
    pl.col('r_government').last().alias('r_government'),
    pl.col('r_travel_and_tourism').last().alias('r_travel_and_tourism'),
    pl.col('r_retail_car_wash').last().alias('r_retail_car_wash'),
    pl.col('r_cpg_beverage_beer_oof').last().alias('r_cpg_beverage_beer_oof'),
    pl.col('r_cpg_beverage_beer_vide').last().alias('r_cpg_beverage_beer_vide'),
    pl.col('r_cpg_beverage_wine_oof').last().alias('r_cpg_beverage_wine_oof'),
    pl.col('r_cpg_beverage_wine_video').last().alias('r_cpg_beverage_wine_video'),
    pl.col('r_finance_credit_cards').last().alias('r_finance_credit_cards'),
    pl.col('r_cpg_cbd_hemp_ingestibles_non_thc').last().alias('r_cpg_cbd_hemp_ingestibles_non_thc'),
    pl.col('r_cpg_non_food_beverage_cannabis_medical').last().alias('r_cpg_non_food_beverage_cannabis_medical'),
    pl.col('r_cpg_non_food_beverage_cannabis_recreational').last().alias('r_cpg_non_food_beverage_cannabis_recreational'),
    pl.col('r_cpg_non_food_beverage_cbd_hemp_non_thc').last().alias('r_cpg_non_food_beverage_cbd_hemp_non_thc'),
    pl.col('r_alcohol_drink_responsibly_message').last().alias('r_alcohol_drink_responsibly_message'),
    pl.col('r_alternative_transportation').last().alias('r_alternative_transportation'),
    pl.col('r_associations_and_npo_anti_smoking').last().alias('r_associations_and_npo_anti_smoking'),
    pl.col('r_automotive_after_market_oil').last().alias('r_automotive_after_market_oil'),
    pl.col('r_cpg_beverage_spirits_ooh').last().alias('r_cpg_beverage_spirits_ooh'),
    pl.col('r_cpg_beverage_spirits_video').last().alias('r_cpg_beverage_spirits_video'),
    pl.col('r_cpg_non_food_beverage_e_cigarette').last().alias('r_cpg_non_food_beverage_e_cigarette'),
    pl.col('r_entertainment_casinos_and_gambling').last().alias('r_entertainment_casinos_and_gambling'),
    pl.col('r_government_political').last().alias('r_government_political'),
    pl.col('r_automotive_electric').last().alias('r_automotive_electric'),
    pl.col('r_recruitment').last().alias('r_recruitment'),
    pl.col('r_restaurants_cdr').last().alias('r_restaurants_cdr'),
    pl.col('r_restaurants_qsr').last().alias('r_restaurants_qsr'),
    pl.col('r_retail_automotive_service').last().alias('r_retail_automotive_service'),
    pl.col('r_retail_grocery').last().alias('r_retail_grocery'),
    pl.col('r_retail_grocerty_with_fuel').last().alias('r_retail_grocerty_with_fuel'),
]

# Add revenue metric totals
for col in revenue_metrics:
    if col in site_scores.columns:
        agg_exprs.append(pl.col(col).sum().alias(f'total_{col}'))

print(f'Aggregating {len(site_scores):,} monthly records to one row per site...')
print(f'Using {len(agg_exprs)} aggregation expressions')

Aggregating 1,475,761 monthly records to one row per site...
Using 75 aggregation expressions


In [14]:
# Sort by site and date; then aggregate
df_sorted = site_scores.sort(['id_gbase', 'date'])
site_agg = df_sorted.group_by('id_gbase').agg(agg_exprs)

# Calculate averages per active month
for col in revenue_metrics:
    total_col = f'total_{col}'
    if total_col in site_agg.columns:
        site_agg = site_agg.with_columns(
            (pl.col(total_col) / pl.col('active_months')).alias(f'avg_monthly_{col}')
        )

print(f'Aggregated to {len(site_agg):,} sites')
print(f'Columns: {len(site_agg.columns)}')

# Show status distribution
status_dist = site_agg.group_by('status').agg(pl.len().alias('count')).sort('count', descending=True)
print(f'\nStatus distribution:')
for row in status_dist.iter_rows(named=True):
    pct = row['count'] / len(site_agg) * 100
    print(f'  {row["status"]:15s} {row["count"]:6,} ({pct:.1f}%)')

Aggregated to 57,675 sites
Columns: 82

Status distribution:
  Active          26,101 (45.3%)
  Temporarily Deactivated 23,374 (40.5%)
  Awaiting Installation  4,417 (7.7%)
  Deactivated      3,094 (5.4%)
  Awaiting Reactivation    627 (1.1%)
  Cancelled           60 (0.1%)
  Awaiting Deactivation      2 (0.0%)


### 2b. Calculate Relative Strength Indicators (Momentum Features)

RS compares short-term vs long-term performance for revenue and impression metrics.  
Values > 1.0 = upward trend, < 1.0 = declining.  
Three horizons capture short/medium/long-term momentum.

In [15]:
# Define calculate_relative_strength(df, metric_col, output_col, short_days, long_days, min_observations)
def calculate_relative_strength(
    df: pl.DataFrame,
    metric_col: str,
    output_col: str,
    short_days: int = 30,
    long_days: int = 90,
    min_observations: int = 3,
) -> pl.DataFrame:
    """Calculate relative strength for a metric (short-term avg / long-term avg)."""
    if metric_col not in df.columns:
        print(f'  Warning: {metric_col} not found, skipping {output_col}')
        return pl.DataFrame({'id_gbase': [], output_col: []})

    if 'date_parsed' not in df.columns:
        df = df.with_columns(pl.col('date').str.to_date().alias('date_parsed'))

    max_date = df['date_parsed'].max()
    short_cutoff = max_date - pl.duration(days=short_days)
    long_cutoff = max_date - pl.duration(days=long_days)

    short_term = (
        df.filter(pl.col('date_parsed') >= short_cutoff)
        .group_by('id_gbase')
        .agg([
            pl.col(metric_col).mean().alias('short_avg'),
            pl.col(metric_col).count().alias('short_count'),
        ])
    )

    long_term = (
        df.filter(pl.col('date_parsed') >= long_cutoff)
        .group_by('id_gbase')
        .agg([
            pl.col(metric_col).mean().alias('long_avg'),
            pl.col(metric_col).count().alias('long_count'),
        ])
    )

    epsilon = 1e-6
    rs_df = short_term.join(long_term, on='id_gbase', how='left').with_columns([
        pl.when(
            (pl.col('short_count') >= min_observations) &
            (pl.col('long_count') >= min_observations)
        )
        .then((pl.col('short_avg') + epsilon) / (pl.col('long_avg') + epsilon))
        .otherwise(None)
        .alias(output_col)
    ]).select(['id_gbase', output_col])

    rs_df = rs_df.with_columns(
        pl.col(output_col).fill_null(1.0).fill_nan(1.0)
    )
    return rs_df


print('Function defined: calculate_relative_strength()')

Function defined: calculate_relative_strength()


In [16]:
# Calculate multi-horizon relative strength features
# Data is MONTHLY, so windows are calibrated for monthly data points:
#   - 95/185 days  → ~3/6 months (short-term)
#   - 185/370 days → ~6/12 months (medium-term)
#   - 370/740 days → ~12/24 months (long-term)
horizons = [(95, 185), (185, 370), (370, 740)]
rs_metrics = [
    ('monthly_impressions', 'Impressions'),
    ('monthly_nvis', 'NVIs'),
    ('revenue', 'Revenue'),
    ('monthly_revenue_per_screen', 'RevenuePerScreen'),
]

# Parse date once
scores_with_date = site_scores.with_columns(pl.col('date').str.to_date().alias('date_parsed'))

all_rs_dfs = []
for short_days, long_days in horizons:
    print(f'\nHorizon: {short_days}/{long_days} days')
    for metric_col, metric_name in rs_metrics:
        output_col = f'rs_{metric_name}_{short_days}_{long_days}'
        rs_df = calculate_relative_strength(
            scores_with_date, metric_col, output_col,
            short_days=short_days, long_days=long_days,
            min_observations=2,
        )
        if len(rs_df) > 0:
            all_rs_dfs.append(rs_df)
            mean_val = rs_df[output_col].mean()
            print(f'  {output_col}: {len(rs_df):,} sites (mean={mean_val:.3f})')

# Join all RS features together
rs_result = all_rs_dfs[0]
for rs_df in all_rs_dfs[1:]:
    rs_result = rs_result.join(rs_df, on='id_gbase', how='full', coalesce=True)

# Fill nulls with 1.0 (neutral)
rs_cols = [col for col in rs_result.columns if col.startswith('rs_')]
for col in rs_cols:
    rs_result = rs_result.with_columns(pl.col(col).fill_null(1.0))

# Join RS features to site aggregates
site_agg = site_agg.join(rs_result, on='id_gbase', how='left')
for col in rs_cols:
    if col in site_agg.columns:
        site_agg = site_agg.with_columns(pl.col(col).fill_null(1.0))

print(f'\nAdded {len(rs_cols)} RS features to site aggregates')
print(f'Site aggregates shape: {site_agg.shape}')


Horizon: 95/185 days
  rs_Impressions_95_185: 31,599 sites (mean=0.962)
  rs_NVIs_95_185: 31,599 sites (mean=0.972)
  rs_Revenue_95_185: 31,599 sites (mean=0.902)
  rs_RevenuePerScreen_95_185: 31,599 sites (mean=0.902)

Horizon: 185/370 days
  rs_Impressions_185_370: 33,303 sites (mean=1.004)
  rs_NVIs_185_370: 33,303 sites (mean=1.021)
  rs_Revenue_185_370: 33,303 sites (mean=1.093)
  rs_RevenuePerScreen_185_370: 33,303 sites (mean=1.093)

Horizon: 370/740 days
  rs_Impressions_370_740: 35,432 sites (mean=0.959)
  rs_NVIs_370_740: 35,432 sites (mean=0.962)
  rs_Revenue_370_740: 35,432 sites (mean=1.044)
  rs_RevenuePerScreen_370_740: 35,432 sites (mean=1.044)

Added 12 RS features to site aggregates
Site aggregates shape: (57675, 94)


---
## Stage 3: Join Geospatial Distance Features

Join the 6 auxiliary distance datasets to the site-level aggregates via `gtvid` ↔ `GTVID`.

In [17]:
# Join distance to nearest_site, interstate, kroger, walmart, mcdonalds, target
site_agg = site_agg.join(
    nearest.select([
        'GTVID',
        'nearest_site',
        pl.col('nearest_site_distance_mi').alias('min_distance_to_nearest_site_mi')
    ]),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join interstate distances
site_agg = site_agg.join(
    interstate,
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Kroger distances
site_agg = site_agg.join(
    kroger.select(['GTVID', 'min_distance_to_kroger_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join McDonald's distances
site_agg = site_agg.join(
    mcdonalds.select(['GTVID', 'min_distance_to_mcdonalds_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Walmart distances
site_agg = site_agg.join(
    walmart.select(['GTVID', 'min_distance_to_walmart_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Target distances
site_agg = site_agg.join(
    target.select(['GTVID', 'min_distance_to_target_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Report join coverage
print('Geospatial feature join coverage:')
for col_name, label in [
    ('nearest_site', 'Nearest site'),
    ('min_distance_to_interstate_mi', 'Interstate'),
    ('min_distance_to_kroger_mi', 'Kroger'),
    ('min_distance_to_mcdonalds_mi', "McDonald's"),
    ('min_distance_to_walmart_mi', 'Walmart'),
    ('min_distance_to_target_mi', 'Target'),
]:
    if col_name in site_agg.columns:
        matched = site_agg.filter(pl.col(col_name).is_not_null()).shape[0]
        pct = matched / len(site_agg) * 100
        print(f'  {label:15s} {matched:6,} / {len(site_agg):,} ({pct:.1f}%)')

print(f'\nPre-cleaned dataset: {site_agg.shape[0]:,} sites × {site_agg.shape[1]} columns')

Geospatial feature join coverage:
  Nearest site    57,651 / 57,675 (100.0%)
  Interstate      57,651 / 57,675 (100.0%)
  Kroger          57,675 / 57,675 (100.0%)
  McDonald's      57,675 / 57,675 (100.0%)
  Walmart         57,675 / 57,675 (100.0%)
  Target          57,675 / 57,675 (100.0%)

Pre-cleaned dataset: 57,675 sites × 102 columns


### Save Pre-cleaned Dataset

In [18]:
# Checkpoint: save pre-cleaned and compressed dataset as
precleaned_path = OUTPUT_DIR / 'site_aggregated_precleaned.parquet'
site_agg.write_parquet(precleaned_path)
size_mb = precleaned_path.stat().st_size / 1024 / 1024
print(f'Saved pre-cleaned checkpoint: {precleaned_path.name} ({size_mb:.1f} MB)')

Saved pre-cleaned checkpoint: site_aggregated_precleaned.parquet (11.9 MB)


**Output:** `site_aggregated_precleaned.parquet` — 57,675 sites × 102 columns (11.9 MB)  
Compressed from ~980 MB of raw CSVs. Ready for exploration in Notebook 2.